# Backprop Ninja

## The Intials

In [14]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [15]:
words = open('names.txt', 'r').read().splitlines()

In [16]:
char = sorted(list(set(''.join(words))))
stoi = { x:i+1 for i, x in enumerate(char)}
stoi['.'] = 0
itos = {x:i for i,x in stoi.items()}
print(stoi)
itos
vocab_size = len(itos)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [17]:
block_size = 3



def build_dataset(words):
    X, Y  = [], []
    for w in words:
        ch = w+'.'
        context = [0] *block_size
        # print(ch)
        for i in ch:
            # print(i)
            ix = stoi[i]
            # print(ix)
            X.append(context)
            Y.append(ix)
            context = context[1:]+[ix]

    X= torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(len(words)*0.8)
n2 = int(len(words)*0.9)

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xtest, Ytest = build_dataset(words[n2:])

torch.Size([182485, 3]) torch.Size([182485])
torch.Size([22822, 3]) torch.Size([22822])
torch.Size([22846, 3]) torch.Size([22846])


In [18]:
def cmp(s, dt, t):
    ex = torch.all (dt ==t.grad).item()
    # print(ex)
    app = torch.allclose(dt, t.grad)
    # print(app)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate : {str(app):5s} | maxdiff: {maxdiff}')

In [19]:
n_embd = 10
n_hidden = 64
g = torch.Generator().manual_seed(2147483647)
C = torch.randn(vocab_size, n_embd)
W1 = torch.randn(n_embd*block_size, n_hidden )*((5/3)/ (n_embd * block_size)**0.5)
b1 = (torch.rand(n_hidden))*0.1
W2 = torch.randn(n_hidden, vocab_size)*0.1
b2 = torch.randn(vocab_size)*0.1
bngain = torch.ones((1,n_hidden))*0.1 + 0.1
bnbias = torch.zeros((1,n_hidden))*0.1
# bnmean_running = torch.ones(1, n_hidden)
# bnstd_running = torch.zeros(1, n_hidden)
parameters = [C, W1, W2, b1, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

4137


In [29]:
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (32,))
Xb, Yb = Xtr[ix], Ytr[ix]
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)
hprebn =  embcat@W1 +b1
# hpreact = (hpreact - hpreact.mean(0, keepdim=True))/hpreact.std(0, keepdim=True)
bnmeani = 1/n*hprebn.mean(0, keepdim = True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim = True)
bnvar_inv = (bnvar +1e-5) **-0.5
bnraw = bndiff*bnvar_inv
hpreact = bngain*bnraw + bnbias

h = torch.tanh(hpreact)

logits = h @ W2 +b2

logit_maxes = logits.max(1, keepdim = True).values
lidx = logits.max(1, keepdim = True).indices
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim= True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

for p in parameters:
    p.grad = None

for t in [logprobs, probs, counts, counts_sum,
         counts_sum_inv, norm_logits, logit_maxes, logits, h, 
         hpreact, bnraw, bnvar_inv, bndiff2, bndiff, hprebn, bnmeani,
            embcat,emb]:
    t.retain_grad()
loss.backward()
loss


tensor(3.2801, grad_fn=<NegBackward0>)

In [22]:
probs.shape
counts.shape
counts_sum_inv.shape

torch.Size([32, 1])

torch.Size([32, 27])

In [24]:
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('normlogits', dnorm_logits, norm_logits)
cmp('logits', dlogits, logits)
cmp('logits_maxes', dlogit_maxes, logit_maxes)

logprobs        | exact: True  | approximate : True  | maxdiff: 0.0
probs           | exact: True  | approximate : True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate : True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate : True  | maxdiff: 0.0
counts          | exact: True  | approximate : True  | maxdiff: 0.0
normlogits      | exact: True  | approximate : True  | maxdiff: 0.0
logits          | exact: False | approximate : True  | maxdiff: 4.423782229423523e-09
logits_maxes    | exact: True  | approximate : True  | maxdiff: 0.0


In [ ]:
probs.shape
logprobs.shape
counts.shape
dcounts.shape
# counts_sum_inv.shape

torch.Size([32, 27])

In [25]:
logits.max(1, keepdim = True)

torch.return_types.max(
values=tensor([[0.2582],
        [0.2749],
        [0.3873],
        [0.3280],
        [0.3585],
        [0.3298],
        [0.6624],
        [0.5301],
        [0.3745],
        [0.3065],
        [0.2728],
        [0.2681],
        [0.2721],
        [0.3991],
        [0.7991],
        [0.2794],
        [0.3049],
        [0.4700],
        [0.3817],
        [0.4133],
        [0.5874],
        [0.3023],
        [0.2729],
        [0.2857],
        [0.5623],
        [0.4135],
        [0.3817],
        [0.2947],
        [0.5207],
        [0.3817],
        [0.3371],
        [0.5713]], grad_fn=<MaxBackward0>),
indices=tensor([[ 6],
        [23],
        [14],
        [23],
        [23],
        [14],
        [23],
        [23],
        [23],
        [ 6],
        [ 0],
        [23],
        [23],
        [14],
        [23],
        [ 6],
        [11],
        [23],
        [ 0],
        [14],
        [23],
        [23],
        [ 3],
        [13],
        [14],
        [

In [ ]:
dlogit_maxes

tensor([[-3.2596e-09],
        [ 3.9581e-09],
        [-4.6566e-10],
        [-4.6566e-10],
        [-6.9849e-10],
        [-2.3283e-10],
        [ 4.1910e-09],
        [-5.8208e-09],
        [-3.2596e-09],
        [-1.8626e-09],
        [-0.0000e+00],
        [-2.0955e-09],
        [ 2.3283e-09],
        [-5.5879e-09],
        [-5.1223e-09],
        [-3.9581e-09],
        [-1.1642e-09],
        [-0.0000e+00],
        [-2.0955e-09],
        [-2.0955e-09],
        [-1.3970e-09],
        [ 2.3283e-09],
        [-2.3283e-09],
        [ 1.8626e-09],
        [-4.1910e-09],
        [ 3.7253e-09],
        [-1.1642e-09],
        [-2.5611e-09],
        [-1.6298e-09],
        [ 1.3970e-09],
        [-6.9849e-10],
        [ 4.6566e-10]], grad_fn=<MulBackward0>)

In [ ]:
logit_maxes

tensor([[0.1912],
        [0.3595],
        [0.3210],
        [0.2146],
        [0.3228],
        [0.2146],
        [0.3447],
        [0.2146],
        [0.1912],
        [0.2976],
        [0.2751],
        [0.3986],
        [0.2385],
        [0.3306],
        [0.1912],
        [0.2045],
        [0.2019],
        [0.3031],
        [0.2146],
        [0.2146],
        [0.2369],
        [0.3593],
        [0.3461],
        [0.1749],
        [0.2043],
        [0.3134],
        [0.2019],
        [0.1915],
        [0.2818],
        [0.1545],
        [0.2437],
        [0.3124]], grad_fn=<MaxBackward0>)

In [ ]:
dlogit_maxes

tensor([[-3.2596e-09],
        [ 3.9581e-09],
        [-4.6566e-10],
        [-4.6566e-10],
        [-6.9849e-10],
        [-2.3283e-10],
        [ 4.1910e-09],
        [-5.8208e-09],
        [-3.2596e-09],
        [-1.8626e-09],
        [-0.0000e+00],
        [-2.0955e-09],
        [ 2.3283e-09],
        [-5.5879e-09],
        [-5.1223e-09],
        [-3.9581e-09],
        [-1.1642e-09],
        [-0.0000e+00],
        [-2.0955e-09],
        [-2.0955e-09],
        [-1.3970e-09],
        [ 2.3283e-09],
        [-2.3283e-09],
        [ 1.8626e-09],
        [-4.1910e-09],
        [ 3.7253e-09],
        [-1.1642e-09],
        [-2.5611e-09],
        [-1.6298e-09],
        [ 1.3970e-09],
        [-6.9849e-10],
        [ 4.6566e-10]], grad_fn=<MulBackward0>)